# Data Science & Database Assignment: Session 11
## Topic: Subqueries (Scalar, Correlated, IN / EXISTS, and Nested Relational Division)

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `subqueries.db`  
**Core Concepts:** Subqueries in `WHERE`, Scalar Subqueries in `SELECT`, Subqueries with `IN`, Correlated vs Non-Correlated Subqueries, Relational Division (`COUNT` & `NOT EXISTS`)  
**Status:** Completed & Pre-Executed  

---

### Understanding Subqueries
A **subquery** (or inner query / nested query) is a `SELECT` statement embedded within another SQL query. Subqueries are categorized into distinct execution paradigms:

```text
+-----------------------+---------------------------------------------------------------------------------+
| Subquery Type         | Mechanism & Use Case                                                            |
+-----------------------+---------------------------------------------------------------------------------+
| Scalar Subquery       | Returns exactly ONE value (1 row, 1 col). Used in SELECT or WHERE comparisons. |
| Column Subquery       | Returns a single column with multiple rows. Used with IN, ANY, ALL.             |
| Correlated Subquery   | References columns from the outer query; re-evaluated for each outer row.       |
| Relational Division   | Compares subsets against a complete universal set (e.g., "products in EVERY").  |
+-----------------------+---------------------------------------------------------------------------------+
```


## Task 1: Subquery in WHERE Clause (Restaurants Above City Average)
> **Problem Statement:** Create a SQL query using a subquery in the `WHERE` clause to find all restaurants from a `Restaurants` table whose average rating is higher than the average rating of all restaurants in the city.

### Conceptual Explanation
To compare each restaurant against the average rating of its own respective city, we use a **Correlated Subquery**:
```sql
WHERE r1.rating > (
    SELECT AVG(r2.rating)
    FROM Restaurants r2
    WHERE r2.city = r1.city
)
```
For each candidate row in `r1`, the database evaluates the average rating of restaurants with the matching city in `r2`. If the individual restaurant's rating exceeds this benchmark, it is returned.


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('subqueries.db')
cursor = conn.cursor()

# Clean table recreation
cursor.execute("DROP TABLE IF EXISTS Restaurants;")

cursor.execute("""
CREATE TABLE Restaurants (
    id INT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    city VARCHAR(50) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL
);
""")

# Populate restaurant data across 3 major metros
restaurants_data = [
    # Mumbai (City Average = 4.00)
    (1, 'Trishna', 'Mumbai', 4.6),
    (2, 'Britannia & Co', 'Mumbai', 4.4),
    (3, 'Bademiya', 'Mumbai', 3.8),
    (4, 'Local Fast Food', 'Mumbai', 3.2),
    # Bangalore (City Average = 4.35)
    (5, 'Truffles', 'Bangalore', 4.7),
    (6, 'Toit Brewpub', 'Bangalore', 4.6),
    (7, 'CTR Shri Sagar', 'Bangalore', 4.5),
    (8, 'Quick Bites', 'Bangalore', 3.6),
    # Delhi (City Average = 4.27)
    (9, 'Bukhara', 'Delhi', 4.8),
    (10, "Karim's", 'Delhi', 4.3),
    (11, 'Paranthe Wali Gali Corner', 'Delhi', 3.7)
]
cursor.executemany("INSERT INTO Restaurants VALUES (?, ?, ?, ?);", restaurants_data)
conn.commit()

# Execute Correlated Subquery:
task1_query = """
SELECT 
    r1.id AS restaurant_id,
    r1.name AS restaurant_name,
    r1.city,
    r1.rating,
    ROUND((SELECT AVG(r2.rating) FROM Restaurants r2 WHERE r2.city = r1.city), 2) AS city_avg_rating
FROM Restaurants r1
WHERE r1.rating > (
    SELECT AVG(r2.rating)
    FROM Restaurants r2
    WHERE r2.city = r1.city
)
ORDER BY r1.city, r1.rating DESC;
"""

df_t1 = pd.read_sql_query(task1_query, conn)
display(df_t1)


,restaurant_id,restaurant_name,city,rating,city_avg_rating
0,5,Truffles,Bangalore,4.7,4.35
1,6,Toit Brewpub,Bangalore,4.6,4.35
2,7,CTR Shri Sagar,Bangalore,4.5,4.35
3,9,Bukhara,Delhi,4.8,4.27
4,10,Karim's,Delhi,4.3,4.27
5,1,Trishna,Mumbai,4.6,4.00
6,2,Britannia & Co,Mumbai,4.4,4.00


## Task 2: Subquery in SELECT Statement (Zomato User Profile Order Summary)
> **Problem Statement:** Write a SQL query that uses a subquery in the `SELECT` statement to display each user's name from a `Users` table along with the total number of orders they have placed from an 'Orders' table, like a summary you might see in a Zomato user profile.

### Conceptual Explanation
A **Scalar Subquery** placed directly within the `SELECT` projection evaluates an aggregated value for each parent row:
```sql
SELECT 
    u.id, 
    u.name,
    (SELECT COUNT(*) FROM Orders o WHERE o.user_id = u.id) AS total_orders
FROM Users u;
```
- For active users, it returns their count of orders.
- For users with zero orders (e.g. newly registered user `Priya Sharma`), the `COUNT(*)` naturally returns `0` rather than `NULL` or omitting the row.


In [2]:
cursor.execute("DROP TABLE IF EXISTS Orders;")
cursor.execute("DROP TABLE IF EXISTS Users;")

cursor.execute("""
CREATE TABLE Users (
    id INT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(100) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Orders (
    id INT PRIMARY KEY,
    user_id INT NOT NULL,
    order_date DATE NOT NULL,
    amount DECIMAL(10, 2) NOT NULL,
    FOREIGN KEY (user_id) REFERENCES Users(id)
);
""")

users_data = [
    (1, 'Atharv Chaudhari', 'atharv@example.com'),
    (2, 'Sneha Patel', 'sneha@example.com'),
    (3, 'Rahul Verma', 'rahul@example.com'),
    (4, 'Priya Sharma', 'priya@example.com')  # Zero orders
]
orders_data = [
    (101, 1, '2026-03-01', 450.00),
    (102, 1, '2026-03-05', 780.00),
    (103, 1, '2026-03-12', 320.00),
    (104, 1, '2026-03-18', 650.00),
    (105, 2, '2026-03-04', 1200.00),
    (106, 2, '2026-03-15', 890.00),
    (107, 3, '2026-03-10', 250.00)
]
cursor.executemany("INSERT INTO Users VALUES (?, ?, ?);", users_data)
cursor.executemany("INSERT INTO Orders VALUES (?, ?, ?, ?);", orders_data)
conn.commit()

task2_query = """
SELECT 
    u.id AS user_id,
    u.name AS user_name,
    (
        SELECT COUNT(*)
        FROM Orders o
        WHERE o.user_id = u.id
    ) AS total_orders,
    COALESCE(
        (
            SELECT SUM(o.amount)
            FROM Orders o
            WHERE o.user_id = u.id
        ), 0.00
    ) AS total_spend
FROM Users u
ORDER BY total_orders DESC;
"""

df_t2 = pd.read_sql_query(task2_query, conn)
display(df_t2)


,user_id,user_name,total_orders,total_spend
0,1,Atharv Chaudhari,4,2200.0
1,2,Sneha Patel,2,2090.0
2,3,Rahul Verma,1,250.0
3,4,Priya Sharma,0,0.0


## Task 3: Subquery with IN Operator (BookMyShow 5-Star Rated Movies)
> **Problem Statement:** Given a `Movies` table and a `Reviews` table, write a SQL query using `IN` with a subquery to list all movies that have at least one review with a rating of 5 stars, as seen in BookMyShow's top-rated section.

### Conceptual Explanation
The `IN` operator filters outer rows based on membership within a dynamically generated set of keys produced by the subquery:
```sql
SELECT * FROM Movies
WHERE id IN (
    SELECT DISTINCT movie_id 
    FROM Reviews 
    WHERE rating = 5
);
```
- The inner query returns the list of unique `movie_id` values that have earned a `5` star review.
- The outer query retrieves full movie details for movies present in that set.
- Movies with ratings exclusively below 5 (e.g. *Race 3*, *Housefull 4*) are excluded.


In [3]:
cursor.execute("DROP TABLE IF EXISTS Reviews;")
cursor.execute("DROP TABLE IF EXISTS Movies;")

cursor.execute("""
CREATE TABLE Movies (
    id INT PRIMARY KEY,
    title VARCHAR(100) NOT NULL,
    genre VARCHAR(50) NOT NULL,
    release_year INT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Reviews (
    id INT PRIMARY KEY,
    movie_id INT NOT NULL,
    reviewer_name VARCHAR(100) NOT NULL,
    rating INT NOT NULL
);
""")

movies_data = [
    (1, 'Inception', 'Sci-Fi', 2010),
    (2, 'Interstellar', 'Sci-Fi', 2014),
    (3, 'Dangal', 'Biography / Sports', 2016),
    (4, 'Oppenheimer', 'Historical Drama', 2023),
    (5, 'Race 3', 'Action', 2018),
    (6, 'Housefull 4', 'Comedy', 2019)
]
reviews_data = [
    (1, 1, 'Amit', 5),
    (2, 1, 'Kunal', 4),
    (3, 2, 'Pooja', 5),
    (4, 3, 'Rohit', 5),
    (5, 4, 'Sanya', 5),
    (6, 5, 'Tanmay', 2),
    (7, 5, 'Vikas', 1),
    (8, 6, 'Ritu', 3)
]
cursor.executemany("INSERT INTO Movies VALUES (?, ?, ?, ?);", movies_data)
cursor.executemany("INSERT INTO Reviews VALUES (?, ?, ?, ?);", reviews_data)
conn.commit()

task3_query = """
SELECT 
    m.id AS movie_id,
    m.title,
    m.genre,
    m.release_year
FROM Movies m
WHERE m.id IN (
    SELECT DISTINCT r.movie_id
    FROM Reviews r
    WHERE r.rating = 5
)
ORDER BY m.release_year DESC;
"""

df_t3 = pd.read_sql_query(task3_query, conn)
display(df_t3)


,movie_id,title,genre,release_year
0,4,Oppenheimer,Historical Drama,2023
1,3,Dangal,Biography / Sports,2016
2,2,Interstellar,Sci-Fi,2014
3,1,Inception,Sci-Fi,2010


## Task 4: Nested Subquery for Relational Division (Flipkart Sellers in Every Category)
> **Problem Statement:** Write a nested SQL query to find the names of all sellers from a `Sellers` table on a Flipkart-style platform who have sold products in every category listed in a `Categories` table.  
> *Hint: Use nested subqueries to compare seller's categories with the complete list of categories.*

---

### In-Depth Mathematical Breakdown (Relational Division)
This is the classic **Relational Division** problem ($R \div S$). We are asked to find sellers $s$ such that:
$$\{ c \in 	ext{Categories} \} \subseteq \{ c \in 	ext{Categories} \mid \exists p \in 	ext{Products} 	ext{ where } p.	ext{seller\_id} = s.	ext{id} \land p.	ext{category\_id} = c.	ext{id} \}$$

#### Approach 4A: Nested Subquery with `COUNT(DISTINCT)`
Compare the distinct count of categories sold by a seller to the total number of categories in the platform:
```sql
SELECT s.id, s.seller_name
FROM Sellers s
WHERE (
    SELECT COUNT(DISTINCT p.category_id)
    FROM Products p
    WHERE p.seller_id = s.id
) = (
    SELECT COUNT(*) FROM Categories
);
```

#### Approach 4B: Classical Double `NOT EXISTS`
Expresses universal quantification via double negation: *"Find sellers for which there does NOT exist a category that the seller has NOT sold"*:
```sql
SELECT s.id, s.seller_name
FROM Sellers s
WHERE NOT EXISTS (
    SELECT c.id FROM Categories c
    WHERE NOT EXISTS (
        SELECT 1 FROM Products p
        WHERE p.seller_id = s.id AND p.category_id = c.id
    )
);
```


In [4]:
cursor.execute("DROP TABLE IF EXISTS Products;")
cursor.execute("DROP TABLE IF EXISTS Categories;")
cursor.execute("DROP TABLE IF EXISTS Sellers;")

cursor.execute("""
CREATE TABLE Sellers (
    id INT PRIMARY KEY,
    seller_name VARCHAR(100) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Categories (
    id INT PRIMARY KEY,
    category_name VARCHAR(100) NOT NULL
);
""")

cursor.execute("""
CREATE TABLE Products (
    id INT PRIMARY KEY,
    seller_id INT NOT NULL,
    category_id INT NOT NULL,
    product_name VARCHAR(100) NOT NULL,
    price DECIMAL(10, 2) NOT NULL
);
""")

categories_data = [(1, 'Electronics'), (2, 'Fashion'), (3, 'Home & Kitchen')]
sellers_data = [
    (1, 'RetailNet India', 4.8),       # Sells in Electronics, Fashion, Home & Kitchen (All 3!)
    (2, 'SuperCom MultiStore', 4.7),   # Sells in Electronics, Fashion, Home & Kitchen (All 3!)
    (3, 'TechWorld Electronics', 4.5), # Sells in Electronics only
    (4, 'StyleStreet Apparels', 4.2)   # Sells in Fashion only
]
products_data = [
    # RetailNet India (Cat 1, 2, 3)
    (101, 1, 1, 'Smartphone Pro', 34999.00),
    (102, 1, 2, 'Denim Jacket', 2499.00),
    (103, 1, 3, 'Non-stick Cookware Set', 3999.00),
    # SuperCom MultiStore (Cat 1, 2, 3)
    (104, 2, 1, 'Noise Cancelling Headphones', 8999.00),
    (105, 2, 2, 'Casual Cotton T-Shirt', 799.00),
    (106, 2, 3, 'Air Purifier', 9499.00),
    # TechWorld Electronics (Cat 1 only)
    (107, 3, 1, 'Mechanical Keyboard', 4500.00),
    # StyleStreet Apparels (Cat 2 only)
    (108, 4, 2, 'Running Shoes', 3200.00)
]

cursor.executemany("INSERT INTO Categories VALUES (?, ?);", categories_data)
cursor.executemany("INSERT INTO Sellers VALUES (?, ?, ?);", sellers_data)
cursor.executemany("INSERT INTO Products VALUES (?, ?, ?, ?, ?);", products_data)
conn.commit()

print("--- Solution 4A: Nested Subquery with COUNT comparison ---")
task4_query_a = """
SELECT 
    s.id AS seller_id,
    s.seller_name,
    s.rating
FROM Sellers s
WHERE (
    SELECT COUNT(DISTINCT p.category_id)
    FROM Products p
    WHERE p.seller_id = s.id
) = (
    SELECT COUNT(*)
    FROM Categories
);
"""
display(pd.read_sql_query(task4_query_a, conn))

print("--- Solution 4B: Classical Double NOT EXISTS Subquery ---")
task4_query_b = """
SELECT 
    s.id AS seller_id,
    s.seller_name,
    s.rating
FROM Sellers s
WHERE NOT EXISTS (
    SELECT c.id
    FROM Categories c
    WHERE NOT EXISTS (
        SELECT 1
        FROM Products p
        WHERE p.seller_id = s.id 
          AND p.category_id = c.id
    )
);
"""
display(pd.read_sql_query(task4_query_b, conn))

conn.close()


--- Solution 4A: Nested Subquery with COUNT comparison ---


,seller_id,seller_name,rating
0,1,RetailNet India,4.8
1,2,SuperCom MultiStore,4.7


--- Solution 4B: Classical Double NOT EXISTS Subquery ---


,seller_id,seller_name,rating
0,1,RetailNet India,4.8
1,2,SuperCom MultiStore,4.7


## Summary & Performance Guidelines

1. **When to use Subquery in WHERE**: Ideal for dynamic thresholding against aggregate metrics (e.g., above average, top percentiles).
2. **When to use Scalar Subquery in SELECT**: Great for ad-hoc KPI cards and user profile rollups where joining would otherwise require complicated multi-table grouping.
3. **`IN` vs `EXISTS`**:
   - `IN` works well when the inner query produces a small, distinct set of scalar IDs.
   - `EXISTS` is preferred when the inner subquery table is massive, as the database short-circuits execution as soon as the first matching record is found.
4. **Relational Division**:
   - The `COUNT(DISTINCT category_id) = (SELECT COUNT(*) FROM Categories)` pattern is intuitive and highly readable.
   - The `NOT EXISTS ... NOT EXISTS` pattern is universally ANSI-SQL compliant and avoids aggregations.
